In [1]:
from __future__ import division
import pandas as pd
import numpy as np
from copy import deepcopy

import warnings
warnings.filterwarnings('ignore')


import py_stringmatching as sm
import py_entitymatching as em
import py_stringsimjoin as ssj

from py_entitymatching.catalog import catalog_manager as cm

import random

# Funzioni utilizzate

In [2]:
######################CLUSTERING######################

import networkx as nx

def ClusterComponentiConnessi(MatchTable, TuttiInodi):

    MatchTable=deepcopy(MatchTable)
    MatchTable.columns=['A','B']

    Singleton = set(TuttiInodi) - set(MatchTable['A']).union(set(MatchTable['B']))

    # Creazione del grafo a partire dagli elementi della MatchTable
    G = nx.Graph()
    for _, row in MatchTable.iterrows():
        G.add_edge(row['A'], row['B'])
#        G.add_edge(row['A'], row['B'], weight=row['sim'])  # Aggiungi il peso (etichetta) basato su 'sim'

    # Aggiungi gli elementi singleton all'insieme dei nodi
    for element in Singleton:
        G.add_node(element)

    # Calcola i componenti connessi (clusters)
    clusters = list(nx.connected_components(G))

    # Creazione del DataFrame dei cluster
    cluster_data = {'ClusterKey': [], 'ClusterElement': []}
    for i, cluster in enumerate(clusters):
        for element in cluster:
            cluster_data['ClusterKey'].append(i + 1)
            cluster_data['ClusterElement'].append(element)

    cluster_df = pd.DataFrame(cluster_data)
    return cluster_df

In [3]:
def VisualizzaCluster2(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

def VisualizzaCluster(Clusters,quanti):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI DELL'IDENTIFICATIVO DEL RECORD
### specificere in quanti ; esempio se il record è S2_123, quanti =2 per ottenere S2
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[:quanti]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS


def _VisualizzaDistribuzioneCluster(Clusters):
    gruppi = Clusters.groupby('ClusterKey')
    conteggio_gruppi = gruppi.size().reset_index(name='NumeroElementiPerCluster')
    Risultato=conteggio_gruppi.groupby('NumeroElementiPerCluster').size().reset_index(name='NumeroCluster')
    print("Numero Elementi", (Risultato['NumeroElementiPerCluster'] * Risultato['NumeroCluster']).sum())
    ClusterMax=conteggio_gruppi[conteggio_gruppi['NumeroElementiPerCluster']==Risultato['NumeroElementiPerCluster'].max()]
    print("Cluster con max numero di elementi:", ClusterMax['ClusterKey'].tolist())

    return Risultato

In [4]:
def CalcolaMatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()

In [5]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=['l_id', 'r_id', "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT['l_id'].isin(MATCH['l_id'])) & (~MT['r_id'].isin(MATCH['r_id']))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['l_id']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['r_id']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [6]:
def Valuta2(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta2(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [7]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
 #   Gold = Gold[['l_id','r_id']]
 #   Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    #Gold = Gold[['l_id','r_id']]
    #Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [8]:
def ValutaBlocking(DA,DB,Block,Gold):
# INPUT : entrambi Block (è il candidate set after blocking)
#        e Gold (Gold Standard) devono essere con due colonne, l_id e r_id
# per avere indipendenza dal nome di queste colonne
# Si suppone che in Block l_id e r_id siano rispettivamente la seconda e la terza colonna
# e che in Gold sia la prima e la seconda
  Block = Block.iloc[:, [1, 2]].copy()
  Gold = Gold.iloc[:, [0, 1]].copy()
  Gold.columns=Block.columns=['l_id','r_id']

  JOIN=pd.merge(Gold, Block)

 # Reduction_Ratio
  RR=1-len(Block)/(len(DA)*len(DB))
 # Pairs Completeness o Recall
  PC = len(JOIN)/len(Gold)
 # Pairs Quality
  PQ = len(JOIN)/len(Block)

  Risultato = pd.DataFrame([(DA.shape[0],DB.shape[0],Block.shape[0],round(RR,4),round(PC,4),round(PQ,4))],
                             columns=['A', 'B', 'BlockSize', 'ReductRatio','PCompletness','PQuality'])

  return Risultato

In [9]:
def IdSOURCES(Sources:list):
  ListaID= []
  for s in Sources.keys():
    ListaID += Sources[s]['id'].to_list()
  return ListaID

# Esercizio Entity Resolution

In [10]:
path='http://dbgroup.ing.unimore.it/EBI/Cluster11/'


src_links = [
path+'A.csv',
path+'B.csv',
path+'C.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

ClusterGoldStandard=pd.read_csv(path+ 'ClusterGoldStandard.csv')

def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS
VisualizzaCluster(ClusterGoldStandard).sort_values('#Record', ascending=False)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
117,118,3,"A,C,B,C",4,"A_261,C_203,B_278,C_558"
45,46,3,"C,A,B",3,"C_275,A_318,B_575"
51,52,3,"A,C,B",3,"A_167,C_546,B_26"
82,83,3,"C,A,B",3,"C_606,A_217,B_210"
100,101,3,"C,B,A",3,"C_22,B_170,A_321"
...,...,...,...,...,...
728,729,1,C,1,C_579
727,728,1,B,1,B_605
726,727,1,C,1,C_318
725,726,1,C,1,C_526


In [12]:
SOURCES['S1']

,Nome,Cognome,DataNascita,Sesso,Nazionalita,CodiceBelfiore,id
0,Massimiliano,Rotg,25/04/1995,M,Italia,A176,A_0
1,Guido,Antonucci,11/06/1974,M,Italia,L736,A_1
2,Giovanni,Grignolo,16/06/1991,M,Italia,A494,A_2
3,Jorge Luis,Millos,22/12/1965,M,Peru,Z611,A_3
4,Sidy,Sandy,31/07/1984,M,Guinea,Z319,A_4
...,...,...,...,...,...,...,...
622,Petre,Sandor,27/10/1959,M,Romania,Z129,A_622
623,Youns,El-Aynaoui,28/09/1974,M,Marocco,Z330,A_623
624,Domenico,Perini,20/04/1975,M,Italia,A376,A_624
625,Adolfo,Lampronti,03/11/1985,M,Italia,C351,A_625


In [11]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

Numero Elementi 1875
Cluster con max numero di elementi: [118]


,NumeroElementiPerCluster,NumeroCluster
0,1,1195
1,2,317
2,3,14
3,4,1


In [13]:
# Metodo di Entity Resolution dato

def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    # BLOCKING by Similarity Join
    Attributi = ['Nome', 'Cognome', 'DataNascita', 'Sesso', 'CodiceBelfiore']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    CandidateDato  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                        l_out_attrs=Attributi,
                                        r_out_attrs=Attributi
                                     )
    CandidateDato=CandidateDato.rename(columns={'l_l_id': 'l_id'})
    CandidateDato=CandidateDato.rename(columns={'r_r_id': 'r_id'})
    CandidateDato=CandidateDato.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.7 + Nome_Nome_lev_sim(ltuple, rtuple)*0.3 > 0.7' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
         # MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

Numero Elementi 1875
Cluster con max numero di elementi: [5, 66, 69, 88, 93]


,NumeroElementiPerCluster,NumeroCluster
0,1,1300
1,2,280
2,3,5


In [ ]:
## Usare le seguenti features
## Nazionalita_Nazionalita_lev_sim(ltuple, rtuple)
## Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)
## Nome_Nome_lev_sim(ltuple, rtuple)
## CodiceBelfiore_CodiceBelfiore_lev_sim(ltuple, rtuple)

# Altre  features si possono creare/visualizzare utilizzando    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)


In [ ]:
# valutazione
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

In [ ]:
# Analisi FN e FP
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])


VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id').head()


Discussione

Lo svolgimento/risposta consiste nella discussione strutturata nei seguenti punti

* Visualizzare la distribuzione dei cluster del ClusterGoldStandard dato

*    Considerando il Metodo di Entity Resolution dato, calcolare i cluster ottenuti e quindi la valutazione rispetto a quelli dati da ClusterGoldStandard in termini di Match Indotti

*    Considerando i FP e/o FN che si ottengono, modificare il Metodo di Entity Resolution dato (sia nella funzione BlockingMatchingRule che nella funzione MatchTableSOURCES dove si decide se usare o meno un global mapping) e discutere miglioramenti
